# Siyaasad News Dataset Cleaning

This notebook cleans and prepares the **Siyaasad** category dataset from two sources:

```text
bbc_siyaasadd_scraped.xlsx
goobjoog_siyaasad_scraped_articles.xlsx
```

The final model target format is:

```text
category: siyaasad | headline: <headline>
```

The model input format is:

```text
analyze somali news article: <clean body>
```

Important idea:

- The category is **not** included in the input because the model must predict it.
- The category is included in the target together with the headline.
- Source-specific cleaning is applied before final category-level checks.

In [2]:
import re
from pathlib import Path

import pandas as pd
import numpy as np

BBC_FILE = Path("raw_dataset/bbc_siyaasadd_scraped.xlsx")
GOOBJOOG_FILE = Path("raw_dataset/goobjoog_siyaasad_scraped_articles.xlsx")

if not BBC_FILE.exists():
    BBC_FILE = Path("/mnt/data/bbc_siyaasadd_scraped.xlsx")

if not GOOBJOOG_FILE.exists():
    GOOBJOOG_FILE = Path("/mnt/data/goobjoog_siyaasad_scraped_articles.xlsx")

OUTPUT_REVIEW_FILE = Path("siyaasad_cleaned_for_review.xlsx")
OUTPUT_FINAL_CSV = Path("siyaasad_model_ready.csv")
OUTPUT_FINAL_XLSX = Path("siyaasad_model_ready.xlsx")
OUTPUT_DROPPED_FILE = Path("siyaasad_dropped_or_review_rows.xlsx")

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_rows", 100)

## 1. Load the source files

The BBC file and Goobjoog file have slightly different schemas, so we load and inspect them separately first.

In [3]:
bbc_raw = pd.read_excel(BBC_FILE)
goobjoog_raw = pd.read_excel(GOOBJOOG_FILE)

print("BBC raw shape:", bbc_raw.shape)
print("Goobjoog raw shape:", goobjoog_raw.shape)

print("\nBBC columns:")
print(bbc_raw.columns.tolist())

print("\nGoobjoog columns:")
print(goobjoog_raw.columns.tolist())

BBC raw shape: (442, 7)
Goobjoog raw shape: (1250, 5)

BBC columns:
['url', 'category', 'page_number', 'listing_url', 'source', 'Headline', 'Body']

Goobjoog columns:
['url', 'headline', 'body', 'source', 'category']


In [4]:
display(bbc_raw.head(3))
display(goobjoog_raw.head(3))

,url,category,page_number,listing_url,source,Headline,Body
0,https://www.bbc.com/somali/articles/cwy8nwr22w8o,siyaasadd,1,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1,BBC Somali,Dowladda Ingiriiska oo ka fikiraysa in Andrew laga saaro nidaamka boqortooyada,"Xigashada Sawirka,Getty Images\nDawladdu waxay ka fikiraysaa soo saarista sharci lagaga saarayo Andrew Mountbatten-Windsor kaa oo ku aadan safka dhaxalka boqortooyada, sida ayB..."
1,https://www.bbc.com/somali/articles/cdx4d1269r0o,siyaasadd,1,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1,BBC Somali,Khilaafka qarsoon ee Imaaraadka iyo Sucuudiga: ma noqon doonaa dhammaadka saaxiibtinnimadooda?,"Xigashada Sawirka,Getty Images\nSanadihii u dambeeyay, jawiga gobolka Bariga dhexe waxa ahaa mid muujinayey khilaaf iyo kala duwanaasho mowqifyada iyo siyaasadaha Boqortooyada ..."
2,https://www.bbc.com/somali/articles/c5y2r9dylpno,siyaasadd,1,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1,BBC Somali,Wiilka Museveni oo ku hanjabay inuu dhufaani doono hoggaamiyaha mucaaradka,"Xigashada Sawirka,Getty Images\nGen Muhoozi Kainerugaba oo ah nin si kulul wax ugu qora barta Twitter-ka (X) isla markaana madax u ah milatariga Uganda, ayaa dadka dhaliila wax..."


,url,headline,body,source,category
0,https://goobjoog.com/2026/02/11/muxuu-ka-dhahay-madaxweyne-xasan-sheekh-maxamuud-guusha-soomaaliya-ee-kursiga-golaha-nabadda-iyo-amniga-midowga-afrika/,Muxuu ka dhahay Madaxweyne Xasan Sheekh Maxamuud guusha Soomaaliya ee kursiga Golaha Nabadda iyo Amniga Midowga Afrika?,"Madaxweynaha Jamhuuriyadda Federaalka Soomaaliya Mudane Xasan Sheekh Maxamuud ayaa hambalyo iyo bogaadin u diray shacabka Soomaaliyeed iyo guud ahaan hay’adaha Qaranka, taas oo...",goobjoog,siyaasad
1,https://goobjoog.com/2026/02/13/wafdi-uu-hoggaaminayo-madaxweyne-xasan-sheekh-oo-addis-ababa-gaaray/,Wafdi uu Hoggaaminayo Madaxweyne Xasan Sheekh oo Addis Ababa Gaaray,"Madaxweynaha Jamhuuriyadda Federaalka ah ee Soomaaliya, Mudane Xasan Sheekh Maxamuud iyo wafdi uu hoggaaminayo ayaa si diirran loogu soo dhoweeyay magaalada Addis Ababa ee caas...",goobjoog,siyaasad
2,https://goobjoog.com/2026/04/29/qiimaha-shidaalka-caalamka-oo-si-kordhaya-iyo-maraykanka-oo-la-sheegay-inuu-sabab-u-yahay/,Qiimaha Shidaalka Caalamka oo si kordhaya iyo Maraykanka oo la sheegay inuu sabab u yahay,"Wararka degdegga ah ee ka imanaya gobolka Bariga Dhexe ayaa sheegaya in qiimaha shidaalka caalamka uu sare u kacay, kaddib warar soo baxaya oo tilmaamaya in Mareykanka qorshe...",goobjoog,siyaasad


## 2. Standardize column names and schema

BBC uses `Headline` and `Body`, while Goobjoog uses `headline` and `body`.  
This step converts both files into a shared schema.

In [5]:
def standardize_schema(df, source_name):
    df = df.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    required_cols = ["url", "headline", "body", "category", "source"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df = df[required_cols + [c for c in df.columns if c not in required_cols]]
    df["source"] = source_name
    df["category"] = "siyaasad"
    return df

bbc = standardize_schema(bbc_raw, "BBC Somali")
goobjoog = standardize_schema(goobjoog_raw, "Goobjoog")

print("BBC standardized shape:", bbc.shape)
print("Goobjoog standardized shape:", goobjoog.shape)

display(bbc.head(2))
display(goobjoog.head(2))

BBC standardized shape: (442, 7)
Goobjoog standardized shape: (1250, 5)


,url,headline,body,category,source,page_number,listing_url
0,https://www.bbc.com/somali/articles/cwy8nwr22w8o,Dowladda Ingiriiska oo ka fikiraysa in Andrew laga saaro nidaamka boqortooyada,"Xigashada Sawirka,Getty Images\nDawladdu waxay ka fikiraysaa soo saarista sharci lagaga saarayo Andrew Mountbatten-Windsor kaa oo ku aadan safka dhaxalka boqortooyada, sida ayB...",siyaasad,BBC Somali,1,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1
1,https://www.bbc.com/somali/articles/cdx4d1269r0o,Khilaafka qarsoon ee Imaaraadka iyo Sucuudiga: ma noqon doonaa dhammaadka saaxiibtinnimadooda?,"Xigashada Sawirka,Getty Images\nSanadihii u dambeeyay, jawiga gobolka Bariga dhexe waxa ahaa mid muujinayey khilaaf iyo kala duwanaasho mowqifyada iyo siyaasadaha Boqortooyada ...",siyaasad,BBC Somali,1,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1


,url,headline,body,category,source
0,https://goobjoog.com/2026/02/11/muxuu-ka-dhahay-madaxweyne-xasan-sheekh-maxamuud-guusha-soomaaliya-ee-kursiga-golaha-nabadda-iyo-amniga-midowga-afrika/,Muxuu ka dhahay Madaxweyne Xasan Sheekh Maxamuud guusha Soomaaliya ee kursiga Golaha Nabadda iyo Amniga Midowga Afrika?,"Madaxweynaha Jamhuuriyadda Federaalka Soomaaliya Mudane Xasan Sheekh Maxamuud ayaa hambalyo iyo bogaadin u diray shacabka Soomaaliyeed iyo guud ahaan hay’adaha Qaranka, taas oo...",siyaasad,Goobjoog
1,https://goobjoog.com/2026/02/13/wafdi-uu-hoggaaminayo-madaxweyne-xasan-sheekh-oo-addis-ababa-gaaray/,Wafdi uu Hoggaaminayo Madaxweyne Xasan Sheekh oo Addis Ababa Gaaray,"Madaxweynaha Jamhuuriyadda Federaalka ah ee Soomaaliya, Mudane Xasan Sheekh Maxamuud iyo wafdi uu hoggaaminayo ayaa si diirran loogu soo dhoweeyay magaalada Addis Ababa ee caas...",siyaasad,Goobjoog


## 3. Source-level quality checks before cleaning

This helps us understand each source before combining.

In [6]:
def word_count(text):
    if pd.isna(text):
        return 0
    return len(str(text).split())

def source_quality_report(df, name):
    print("=" * 80)
    print(name)
    print("Rows:", len(df))
    print("Missing values:")
    print(df[["url", "headline", "body", "category", "source"]].isna().sum())
    print("\nCategory counts:")
    print(df["category"].value_counts(dropna=False))
    print("\nSource counts:")
    print(df["source"].value_counts(dropna=False))
    print("\nDuplicate URLs:", df["url"].astype(str).str.strip().duplicated().sum())
    print("Duplicate headlines:", df["headline"].astype(str).str.strip().duplicated().sum())
    print("Duplicate bodies:", df["body"].astype(str).str.strip().duplicated().sum())
    print("\nBody word count:")
    print(df["body"].apply(word_count).describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]))

source_quality_report(bbc, "BBC Somali - raw")
source_quality_report(goobjoog, "Goobjoog - raw")

BBC Somali - raw
Rows: 442
Missing values:
url         0
headline    0
body        0
category    0
source      0
dtype: int64

Category counts:
category
siyaasad    442
Name: count, dtype: int64

Source counts:
source
BBC Somali    442
Name: count, dtype: int64

Duplicate URLs: 0
Duplicate headlines: 1
Duplicate bodies: 2

Body word count:
count     442.000000
mean      747.000000
std       292.952443
min        20.000000
1%         88.640000
5%        131.550000
10%       505.100000
25%       604.000000
50%       716.500000
75%       871.750000
90%      1103.900000
95%      1197.600000
99%      1660.740000
max      2256.000000
Name: body, dtype: float64
Goobjoog - raw
Rows: 1250
Missing values:
url         0
headline    0
body        0
category    0
source      0
dtype: int64

Category counts:
category
siyaasad    1250
Name: count, dtype: int64

Source counts:
source
Goobjoog    1250
Name: count, dtype: int64

Duplicate URLs: 0
Duplicate headlines: 1
Duplicate bodies: 2

Body word cou

## 4. Combine Siyaasad sources

We combine after schema standardization, while still keeping the `source` column so source-specific cleaning can be applied.

In [7]:
df = pd.concat([bbc, goobjoog], ignore_index=True)

df["url"] = df["url"].astype(str).str.strip()
df["headline"] = df["headline"].astype(str).str.strip()
df["body"] = df["body"].astype(str).str.strip()
df["category"] = "siyaasad"

print("Combined shape:", df.shape)
print("\nSource counts:")
print(df["source"].value_counts())
print("\nCategory counts:")
print(df["category"].value_counts())
df.head()

Combined shape: (1692, 7)

Source counts:
source
Goobjoog      1250
BBC Somali     442
Name: count, dtype: int64

Category counts:
category
siyaasad    1692
Name: count, dtype: int64


,url,headline,body,category,source,page_number,listing_url
0,https://www.bbc.com/somali/articles/cwy8nwr22w8o,Dowladda Ingiriiska oo ka fikiraysa in Andrew laga saaro nidaamka boqortooyada,"Xigashada Sawirka,Getty Images\nDawladdu waxay ka fikiraysaa soo saarista sharci lagaga saarayo Andrew Mountbatten-Windsor kaa oo ku aadan safka dhaxalka boqortooyada, sida ayB...",siyaasad,BBC Somali,1.0,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1
1,https://www.bbc.com/somali/articles/cdx4d1269r0o,Khilaafka qarsoon ee Imaaraadka iyo Sucuudiga: ma noqon doonaa dhammaadka saaxiibtinnimadooda?,"Xigashada Sawirka,Getty Images\nSanadihii u dambeeyay, jawiga gobolka Bariga dhexe waxa ahaa mid muujinayey khilaaf iyo kala duwanaasho mowqifyada iyo siyaasadaha Boqortooyada ...",siyaasad,BBC Somali,1.0,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1
2,https://www.bbc.com/somali/articles/c5y2r9dylpno,Wiilka Museveni oo ku hanjabay inuu dhufaani doono hoggaamiyaha mucaaradka,"Xigashada Sawirka,Getty Images\nGen Muhoozi Kainerugaba oo ah nin si kulul wax ugu qora barta Twitter-ka (X) isla markaana madax u ah milatariga Uganda, ayaa dadka dhaliila wax...",siyaasad,BBC Somali,1.0,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1
3,https://www.bbc.com/somali/articles/c78lz73w1ngo,Hoggaamiye Afrikaan ah oo dadkiisa u sheegay inay iska illaawaan dimuqraadiyad,Hoggaamiyaha Burkino Faso ayaa waraysi uu siiyay talefishinka dowladda ku tilmaamay Dimuqraadiyada mid ''dhimasho badan leh'' ayna tahay shacabka Burkino Faso inay ''iska ilaaw...,siyaasad,BBC Somali,1.0,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1
4,https://www.bbc.com/somali/articles/cze0lkwjj48o,Trump oo amray sii deynta xogo ku saabsan makhluuqaadka aan la aqoon ee UFOs,"Xigashada Sawirka,Getty Images\nMadaxweynaha Maraykanka Donald Trump ayaa ku dhawaaqay inuu amri doono hay'adaha dowladda inay sii daayaan dukumentiyo ku saabsan UFO-yada iyo m...",siyaasad,BBC Somali,1.0,https://www.bbc.com/somali/topics/cwr9jry737xt?page=1


## 5. Inspect repeated line patterns

This cell shows repeated body lines. Repeated lines often reveal boilerplate, image credits, ads, footers, and social sharing text.

In [8]:
from collections import Counter

def normalize_newlines(text):
    if pd.isna(text):
        return ""
    text = str(text).replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text

def get_nonempty_lines(text):
    text = normalize_newlines(text)
    return [line.strip() for line in text.splitlines() if line.strip()]

def repeated_lines_report(df, source_name, top_n=40):
    lines = Counter()
    subset = df[df["source"] == source_name]
    for text in subset["body"]:
        lines.update(get_nonempty_lines(text))

    print("=" * 80)
    print(source_name)
    print("Rows:", len(subset))
    print("Top repeated lines:")
    for line, count in lines.most_common(top_n):
        if count > 1:
            print(f"{count:>4} | {line[:220]}")

repeated_lines_report(df, "BBC Somali")
repeated_lines_report(df, "Goobjoog")

BBC Somali
Rows: 442
Top repeated lines:
 442 | ©2026 BBC. BBC masuul kama ahan macluumadka bogagga kale ee dibadda.Akhri xogta ku saabsan sida aan u abaarno bogagga dibadda.
 428 | End of content
 408 | End of Ugu akhris badan
 369 | Warbixinada qotada dheer iyo wararka BBC Somali oo toos kuugu imanaaya WhatsApp.
 369 | Halkaan kaga soo biir
 369 | Dhamaadka xayeysiinta
 255 | Xigashada Sawirka,Getty Images
  98 | Xigashada Sawirka,Reuters
  55 | Xigashada Sawirka,SOCIAL
  31 | Xigashada Sawirka,AFP
  29 | Xigashada Sawirka,Facebook
  28 | Xigashada Sawirka,Villa Somalia
  26 | Xigashada Sawirka,EPA
  22 | Xigashada Sawirka,X
  22 | Xigashada Sawirka,Social
  19 | Xigashada Sawirka,Google
  18 | Xigashada Sawirka,SNTV
  18 | Xigashada Sawirka,.
  16 | Xigashada Sawirka,Social Media
  11 | Xigashada Sawirka,OPM
  10 | Xigashada Sawirka,FACEBOOK
  10 | Xigashada Sawirka,SONNA
   9 | Xigashada Sawirka,AFP via Getty Images
   8 | Xigashada Sawirka,Baraha Bulshada
   7 | Xigashada Sawirka,

## 6. Inspect body beginnings and endings

This is useful for discovering noise that appears at the start or end of articles.

In [9]:
def show_body_samples(df, source_name, n=3):
    subset = df[df["source"] == source_name].copy()

    print("=" * 80)
    print(source_name, "- first body samples")
    for text in subset["body"].dropna().head(n):
        print("-" * 80)
        print(str(text)[:1200])

    print("\n" + "=" * 80)
    print(source_name, "- ending body samples")
    for text in subset["body"].dropna().tail(n):
        print("-" * 80)
        print(str(text)[-1200:])

show_body_samples(df, "BBC Somali")
show_body_samples(df, "Goobjoog")

BBC Somali - first body samples
--------------------------------------------------------------------------------
Xigashada Sawirka,Getty Images
Dawladdu waxay ka fikiraysaa soo saarista sharci lagaga saarayo Andrew Mountbatten-Windsor kaa oo ku aadan safka dhaxalka boqortooyada, sida ayBBC  fahansan tahay.
Tallaabadan, oo ka hortagaysa Mountbatten-Windsor inuu weligiis noqdo boqor, ayaa imaan doonta ka dib markii baaritaanka booliisku soo gabagaboobo.
Boqorka walaalkii ayaa weli ku jira safka sideedaad ee carshiga inkasta oo laga xayuubiyay magacyadiisii, oo ay ku jiraan "amiir", bishii Oktoobar ee la soo dhaafay iyada oo cadaadis lagu saarayo xiriirka uu la leeyahay maalgeliyaha carruurta xadgudubka kula kacay ee Jeffrey Epstein.
Fiidnimadii Khamiista, Mountbatten-Windsor waxaa lagu sii daayay baaris 11 saacadood ka dib markii loo xiray tuhunka anshax-xumo ee xafiiska dadweynaha. Waxa uu si joogto ah oo adag uu u diiday in uu wax gaf ah sameeyay.
Jimcihii baabuur booliis ah oo aan cal

## 7. Cleaning functions

BBC repeated noise found in this dataset includes:

```text
Xigashada Sawirka,...
End of content
End of Ugu akhris badan
Warbixinada qotada dheer...
Halkaan kaga soo biir
Dhamaadka xayeysiinta
©2026 BBC...
```

Goobjoog has less footer noise in the Siyaasad file, but we still remove obvious boilerplate lines such as `Goobjoog News`, `Dhageyso`, and empty/placeholder lines.

In [11]:
def normalize_basic_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\u200b", "")
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")

    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


BBC_REMOVE_LINE_PATTERNS = [
    r"^Xigashada Sawirka\s*,?.*$",
    r"^Qoraalka Sawirka\s*,?.*$",
    r"^End of content$",
    r"^End of Ugu akhris badan$",
    r"^Dhamaadka xayeysiinta$",
    r"^Halkaan kaga soo biir$",
    r"^Warbixinada qotada dheer.*$",
    r"^©\d{4}\s+BBC\..*$",
    r"^BBC masuul kama ahan.*$",
    r"^Fadlanhalkan riix.*$",
    r"^Macluumaadkan waxaa daabacay X\..*$",
    r"^Waxaan dalbanayenaa fasaxaaga.*$",
    r"^Dhammaadka X boggan$",
    r"^Macluumaadkan lama heli karo$",
    r"^Calaamadaha:$",
]

GOOBJOOG_REMOVE_LINE_PATTERNS = [
    r"^Goobjoog News$",
    r"^Googjoog News$",
    r"^Dhageyso$",
    r"^Halkaan hoose ka dhageyso:?$",
    r"^Halkaan ka Akhriso:?$",
    r"^Halkan ka Akhriso:?$",
    r"^Akhriso:?$",
    r"^________+$",
    r"^_+$",
    r"^\.$",
]


def remove_noise_lines(text, source):
    text = normalize_basic_text(text)

    source_lower = str(source).strip().lower()
    if source_lower == "bbc somali":
        patterns = BBC_REMOVE_LINE_PATTERNS
    elif source_lower == "goobjoog":
        patterns = GOOBJOOG_REMOVE_LINE_PATTERNS
    else:
        patterns = []

    cleaned_lines = []
    for line in text.splitlines():
        line = normalize_basic_text(line)

        if not line:
            continue

        should_remove = any(
            re.search(pattern, line, flags=re.IGNORECASE)
            for pattern in patterns
        )

        if not should_remove:
            cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)

    return cleaned.strip()


def clean_headline(text):
    text = normalize_basic_text(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip(" -–—|")
    return text

## 8. Apply cleaning

The raw columns remain unchanged. Cleaned text is stored in new columns.

In [12]:
df["body_clean"] = df.apply(lambda row: remove_noise_lines(row["body"], row["source"]), axis=1)
df["headline_clean"] = df["headline"].apply(clean_headline)

df["body_word_count_raw"] = df["body"].apply(lambda x: len(normalize_basic_text(x).split()))
df["body_word_count_clean"] = df["body_clean"].apply(lambda x: len(str(x).split()))
df["headline_word_count_clean"] = df["headline_clean"].apply(lambda x: len(str(x).split()))

df[["source", "headline_clean", "body_word_count_raw", "body_word_count_clean"]].head()

,source,headline_clean,body_word_count_raw,body_word_count_clean
0,BBC Somali,Dowladda Ingiriiska oo ka fikiraysa in Andrew laga saaro nidaamka boqortooyada,581,535
1,BBC Somali,Khilaafka qarsoon ee Imaaraadka iyo Sucuudiga: ma noqon doonaa dhammaadka saaxiibtinnimadooda?,1543,1488
2,BBC Somali,Wiilka Museveni oo ku hanjabay inuu dhufaani doono hoggaamiyaha mucaaradka,1764,1705
3,BBC Somali,Hoggaamiye Afrikaan ah oo dadkiisa u sheegay inay iska illaawaan dimuqraadiyad,717,667
4,BBC Somali,Trump oo amray sii deynta xogo ku saabsan makhluuqaadka aan la aqoon ee UFOs,676,627


## 9. Compare body length before and after cleaning

A small drop is expected because we remove image credits, footers, and repeated website text.  
A very large drop should be manually inspected.

In [13]:
print("Raw body length:")
print(df["body_word_count_raw"].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]))

print("\nClean body length:")
print(df["body_word_count_clean"].describe(percentiles=[.01, .05, .10, .25, .5, .75, .90, .95, .99]))

print("\nBy source:")
print(df.groupby("source")["body_word_count_clean"].describe())

Raw body length:
count    1692.000000
mean      393.356383
std       348.799392
min        20.000000
1%         87.910000
5%        114.000000
10%       129.000000
25%       164.750000
50%       236.500000
75%       578.000000
90%       838.200000
95%      1049.900000
99%      1610.000000
max      4348.000000
Name: body_word_count_raw, dtype: float64

Clean body length:
count    1692.000000
mean      380.128842
std       334.310576
min         0.000000
1%         83.730000
5%        111.000000
10%       129.000000
25%       164.000000
50%       236.500000
75%       538.250000
90%       790.500000
95%      1010.900000
99%      1595.440000
max      4348.000000
Name: body_word_count_clean, dtype: float64

By source:
             count        mean         std   min     25%    50%    75%     max
source                                                                        
BBC Somali   442.0  696.384615  286.102096   0.0  551.75  663.5  818.0  2172.0
Goobjoog    1250.0  268.300800  272.9529

## 10. Verify source-specific noise removal

The counts should ideally be zero for BBC boilerplate patterns.  
For Goobjoog, some real article sentences may mention "Goobjoog News", so do not remove embedded mentions blindly unless they are standalone noise lines.

In [14]:
noise_checks = {
    "BBC image credit": r"(?m)^Xigashada Sawirka",
    "BBC content marker": r"(?m)^End of content$",
    "BBC popular marker": r"(?m)^End of Ugu akhris badan$",
    "BBC footer": r"©\d{4}\s+BBC",
    "BBC WhatsApp promo": r"Warbixinada qotada dheer",
    "Goobjoog standalone line": r"(?m)^Goobjoog News$",
    "Dhageyso line": r"(?m)^Dhageyso$",
}

for label, pattern in noise_checks.items():
    count = df["body_clean"].str.contains(pattern, case=False, regex=True, na=False).sum()
    print(f"{label}: {count}")

BBC image credit: 0
BBC content marker: 0
BBC popular marker: 0
BBC footer: 0
BBC WhatsApp promo: 0
Goobjoog standalone line: 0
Dhageyso line: 0


## 11. Check records that became empty or too short

BBC has a few video/audio-style pages where the body contains only a BBC footer.  
Those records are not useful for model training unless you manually re-scrape the real article text.

In [15]:
empty_or_missing = df[
    (df["body_clean"].str.strip() == "") |
    (df["headline_clean"].str.strip() == "")
].copy()

print("Rows with empty clean body/headline:", len(empty_or_missing))
display(empty_or_missing[["url", "source", "headline_clean", "body_word_count_raw", "body", "body_clean"]])

Rows with empty clean body/headline: 3


,url,source,headline_clean,body_word_count_raw,body,body_clean
246,https://www.bbc.com/somali/articles/c51yelxl00wo,BBC Somali,"Dhageyso: Abiy Axmed ""Khudaarteenna nooga bedesha badda, meelo kalana ha wareeginina ee anaga nala hadla""",20,©2026 BBC. BBC masuul kama ahan macluumadka bogagga kale ee dibadda.Akhri xogta ku saabsan sida aan u abaarno bogagga dibadda.,
379,https://www.bbc.com/somali/articles/c3g293l08l5o,BBC Somali,Ciidamada Soomaaliya ma waxay u diyaarsan yihiin in ay ATMIS kala wareegaan saldhigyada haray?,20,©2026 BBC. BBC masuul kama ahan macluumadka bogagga kale ee dibadda.Akhri xogta ku saabsan sida aan u abaarno bogagga dibadda.,
421,https://www.bbc.com/somali/articles/ckvz4ejwelzo,BBC Somali,Daawo: Maxay Puntland iyo Dawladda Federaalka marwalba isku qabtaan?,20,©2026 BBC. BBC masuul kama ahan macluumadka bogagga kale ee dibadda.Akhri xogta ku saabsan sida aan u abaarno bogagga dibadda.,


In [16]:
short_articles = df[
    (df["body_word_count_clean"] > 0) &
    (df["body_word_count_clean"] < 100)
].copy()

print("Clean articles under 100 words:", len(short_articles))
display(short_articles[["url", "source", "headline_clean", "body_word_count_clean", "body_clean"]].sort_values("body_word_count_clean").head(50))

Clean articles under 100 words: 43


,url,source,headline_clean,body_word_count_clean,body_clean
1164,https://goobjoog.com/2025/12/23/degdeg/,Goobjoog,Degdeg,30,"Turkiga oo xaqiijiyay in dhinteen dhammaan dadkii saarnaa diyaaraddii la socday taliyaha ciidamada Liibiya General Mohamed Ali Ahmed, waxayna tirada dadkaasi ka koobnayd 5 qof ..."
752,https://goobjoog.com/2025/12/13/golaha-wasiiradda-hirshabeelle-oo-ansixiyay-laba-sharci-oo-muhiim-ah/,Goobjoog,Golaha Wasiiradda Hirshabeelle oo Ansixiyay laba Sharci oo muhiim ah,52,Ku-simaha Madaxweynaha maamulka Hirshabeelle ahna Madaxweyne ku-xigeenka Mudane Yuusuf Axmed Hagar (Dabageed) ayaa shir-guddoomiyey kulanka Golaha Wasiirrada Dowladda Hirshabee...
702,https://goobjoog.com/2026/01/06/%f0%9d%90%96%f0%9d%90%9a%f0%9d%90%ac%f0%9d%90%a2%f0%9d%90%a2%f0%9d%90%ab%f0%9d%90%a4%f0%9d%90%9a-%f0%9d%90%80%f0%9d%90%ab%f0%9d%90%ab%f0%9d%90%a...,Goobjoog,𝐖𝐚𝐬𝐢𝐢𝐫𝐤𝐚 𝐀𝐫𝐫𝐢𝐦𝐚𝐡𝐚 𝐃𝐢𝐛𝐚𝐝𝐝𝐚 𝐘𝐚𝐡𝐮𝐮𝐝𝐝𝐚 𝐆𝐢𝐝𝐞𝐨𝐧 𝐒𝐚'𝐚𝐫 𝐚𝐲𝐚𝐚 𝐤𝐚 𝐝𝐞𝐠𝐚𝐲 𝐦𝐚𝐠𝐚𝐚𝐥𝐚𝐝𝐚 𝐇𝐚𝐫𝐠𝐞𝐲𝐬𝐚,56,"Warbaahinta Yahuuda ayaa xaqiijisay in Wafdiga uu hoggaaminayo Wasiirka Arrimaha dibadda Israail uu tegay magaalada Hargeysa, isagoo kulamo la leh Madaxda Somaliland.\nAmniga M..."
105,https://www.bbc.com/somali/articles/cn0w01n87ljo,BBC Somali,Dhegeyso: Faahfaahin ku saabsan shirka golaha wadatashiga ee Muqdisho ka furmay,63,Magaalada Muqdisho waxaa galinkii dambe ka furmay shirka madaxda maamul goboleedyada iyo dawladda federaalka ee loo yaqaanno shirka golaha wadatashiga.\nShirkan ayaa hore u baa...
309,https://www.bbc.com/somali/articles/cgevdw0yl9zo,BBC Somali,Daawo: Xaggee ku dambeeyey shirkii Puntland ay gogoshiisa dhigtay magaalada Garowe?,67,"Muxuu salka ku hayaa khilaafka iyo xiriir xumada raagtey ee u dhexaya Puntland iyo dowladda federaalka ah ee Soomaaliya, xaggeese ku dambeeyey shirkii Puntland ay gogoshiisa dh..."
812,https://goobjoog.com/2025/12/25/tirinta-codadka-doorashada-oo-ka-socota-xarunta-guddiga-doorashooyinka/,Goobjoog,Tirinta codadka doorashada oo ka socota xarunta Guddiga Doorashooyinka,71,"Xaaladda doorashada dalka ayaa galay weji cusub ka dib markii si rasmi ah loo soo xiray goobihii codbixinta, waxaana hadda si habsami leh u socota tirinta codadka ay dhiibtaan ..."
375,https://www.bbc.com/somali/articles/c28v4gn3mmno,BBC Somali,Wixii kala qabsaday Caasho Geelle iyo aabaheed markii ay surwaal xiratay,71,"Caasho Geelle Diiriye waa mid ka mid ah hablaha ugu caansan siyaasadda Soomaalida, iyada oo noqotay gabadhii ugu horreysay ee xildhibaan iyo wasiir ka noqota dowlad goboleedka ..."
403,https://www.bbc.com/somali/articles/c4n047l23rno,BBC Somali,Dhegeyso wuxuu Xamse ka doonay Dooxa iyo kulankii uu la yeeshay Farmaajo iyo Raysal wasaaraha Qadar,72,Ra'iisul wasaaraha Soomaaliya Xamse Cabdi Barre oo jooga Magaalada Dooxa ee dalka Qadar ayaa shalay kulan gaar ah la yeeshay madaxweynihii hore ee Soomaaliya Maxamed Cabdullaah...
404,https://www.bbc.com/somali/articles/cjq42wxjp53o,BBC Somali,"Dhegeyso: Barre Hiiraale: ""Saraakiil Itoobiyaan ah ayaa iigu hanjabay in anigoo katiinadeysan ay iga saari doonaan Doolow""",72,Barre Adan Shire (Hiiraale) oo ka mid ah siyaasiyiinta ka soo jeeda gobolka Gedo oo muddooyinkii ugu dambeeyayna ku sugnaa gobolkaas ayaa sheegay inay hanjabaad uga timid ciida...
1090,https://goobjoog.com/2026/03/18/sucuudiga-oo-ku-dhawaaqay-in-jimcaha-ay-tahay-maalinta-koowaad-ee-ciidul-fidriga/,Goobjoog,Sucuudiga oo ku dhawaaqay in Jimcaha ay tahay Maalinta Koowaad ee Ciidul Fidriga,73,"Maamulka Haramain Sharifain ee dalka Boqortooyada Sucuudiga ayaa si rasmi ah u ku dhawaaqay in caawa la waayay aragtida bishii Shawaal ee sanadka 1447, taas oo ka dhigan in bis..."


## 12. Duplicate checks

In [17]:
print("Duplicate URLs:", df["url"].duplicated().sum())
print("Duplicate clean headlines:", df["headline_clean"].duplicated().sum())
print("Duplicate clean bodies:", df["body_clean"].duplicated().sum())

duplicate_urls = df[df["url"].duplicated(keep=False)].sort_values("url")
duplicate_bodies = df[df["body_clean"].duplicated(keep=False)].sort_values("body_clean")
duplicate_headlines = df[df["headline_clean"].duplicated(keep=False)].sort_values("headline_clean")

print("\nDuplicate URL rows:", len(duplicate_urls))
print("Duplicate body rows:", len(duplicate_bodies))
print("Duplicate headline rows:", len(duplicate_headlines))

Duplicate URLs: 0
Duplicate clean headlines: 2
Duplicate clean bodies: 5

Duplicate URL rows: 0
Duplicate body rows: 9
Duplicate headline rows: 4


In [18]:
display(duplicate_bodies[["url", "source", "headline_clean", "body_word_count_clean", "body_clean"]].head(30))

,url,source,headline_clean,body_word_count_clean,body_clean
246,https://www.bbc.com/somali/articles/c51yelxl00wo,BBC Somali,"Dhageyso: Abiy Axmed ""Khudaarteenna nooga bedesha badda, meelo kalana ha wareeginina ee anaga nala hadla""",0,
379,https://www.bbc.com/somali/articles/c3g293l08l5o,BBC Somali,Ciidamada Soomaaliya ma waxay u diyaarsan yihiin in ay ATMIS kala wareegaan saldhigyada haray?,0,
421,https://www.bbc.com/somali/articles/ckvz4ejwelzo,BBC Somali,Daawo: Maxay Puntland iyo Dawladda Federaalka marwalba isku qabtaan?,0,
904,https://goobjoog.com/2026/03/24/maxkamadda-ciidanka-soomaaliya-xukun-ku-riday-ninkii-basaasay-feeryahanad-ramla-cali-iyo-kooxdiisa-2/,Goobjoog,Maxkamadda ciidanka Soomaaliya Xukun ku riday Ninkii Basaasay Feeryahanad Ramla Cali iyo Kooxdiisa,386,"Cabdisalaam Maxamed Xasan, Cabdullahi Cusmaan Maxamed, Cabdinaasir Cumar Axmed Subeyr, Faadumo Cismaan Subeyr Cali iyo Leylo Seef Faarax Axmed, Ayaa lagu Eedeeyay in ay siyaabo..."
1188,https://goobjoog.com/2026/03/24/maxkamadda-ciidanka-soomaaliya-xukun-ku-riday-ninkii-basaasay-feeryahanad-ramla-cali-iyo-kooxdiisa/,Goobjoog,Maxkamadda ciidanka Soomaaliya Xukun ku riday Ninkii Basaasay Feeryahanad Ramla Cali iyo Kooxdiisa,386,"Cabdisalaam Maxamed Xasan, Cabdullahi Cusmaan Maxamed, Cabdinaasir Cumar Axmed Subeyr, Faadumo Cismaan Subeyr Cali iyo Leylo Seef Faarax Axmed, Ayaa lagu Eedeeyay in ay siyaabo..."
408,https://www.bbc.com/somali/articles/cd1mxrvxpggo,BBC Somali,Kulanka Ankara iyo wixii ay ka wada hadleen Shariif iyo Farmaajo,101,"Labo madaxweyne oo hore, Maxamed Cabdulaahi Farmaajo iyo Shariif Sheekh Axmed ayaa toddobaadkan ku kulmay magaalada Ankara ee xarunta u ah dowladda Turkey.\nWarar kala duwan ay..."
415,https://www.bbc.com/somali/articles/c4nqkpy9jpzo,BBC Somali,Kulanka Ankara iyo wixii ay ka wada hadleen Shariif iyo Farmaajo,101,"Labo madaxweyne oo hore, Maxamed Cabdulaahi Farmaajo iyo Shariif Sheekh Axmed ayaa toddobaadkan ku kulmay magaalada Ankara ee xarunta u ah dowladda Turkey.\nWarar kala duwan ay..."
1128,https://goobjoog.com/2025/10/25/ma-muqdisho-ayaa-keyn-noqatay-mise-geelii-ayaa-dab-joog-noqday/,Goobjoog,Ma Muqdisho ayaa keyn noqatay mise geelii ayaa dab joog noqday ?,299,"Waddooyinka caasimadda Soomaaliya ee Muqdisho, oo markii hore la ciir ciirayay lo'da, dameeraha iyo ariga buux dhaafiyay, ayaa haatan u muuqda kuwo isu beddelay goobo lagu dhaq..."
1166,https://goobjoog.com/2025/11/24/geelii-dab-joogta-kamid-noqday-iyo-waddooyinka-muqdisho-oo-noqday-sida-keymo-geela-lagu-dhaqdo/,Goobjoog,geelii dab-joogta kamid noqday iyo waddooyinka Muqdisho oo noqday sida keymo geela lagu dhaqdo.,299,"Waddooyinka caasimadda Soomaaliya ee Muqdisho, oo markii hore la ciir ciirayay lo'da, dameeraha iyo ariga buux dhaafiyay, ayaa haatan u muuqda kuwo isu beddelay goobo lagu dhaq..."


## 13. Create review flags

These flags help you manually review suspicious rows before training.

In [20]:
df["needs_review"] = False
df["review_reason"] = ""

def add_review_flag(condition, reason):
    global df
    df.loc[condition, "needs_review"] = True
    df.loc[condition, "review_reason"] = (
        df.loc[condition, "review_reason"].astype(str).str.strip()
        + "; "
        + reason
    ).str.strip("; ")

add_review_flag(df["body_clean"].str.strip().eq(""), "empty_body_after_cleaning")
add_review_flag(df["headline_clean"].str.strip().eq(""), "empty_headline_after_cleaning")
add_review_flag(df["body_word_count_clean"] < 50, "body_under_50_words")
add_review_flag(df["headline_word_count_clean"] < 3, "headline_under_3_words")
add_review_flag(df["body_clean"].duplicated(keep=False), "duplicate_body_clean")
add_review_flag(df["url"].duplicated(keep=False), "duplicate_url")

print(df["needs_review"].value_counts())
display(df[df["needs_review"]][["url", "source", "headline_clean", "body_word_count_clean", "review_reason", "body_clean"]].head(100))

needs_review
False    1681
True       11
Name: count, dtype: int64


,url,source,headline_clean,body_word_count_clean,review_reason,body_clean
246,https://www.bbc.com/somali/articles/c51yelxl00wo,BBC Somali,"Dhageyso: Abiy Axmed ""Khudaarteenna nooga bedesha badda, meelo kalana ha wareeginina ee anaga nala hadla""",0,empty_body_after_cleaning; body_under_50_words; duplicate_body_clean,
379,https://www.bbc.com/somali/articles/c3g293l08l5o,BBC Somali,Ciidamada Soomaaliya ma waxay u diyaarsan yihiin in ay ATMIS kala wareegaan saldhigyada haray?,0,empty_body_after_cleaning; body_under_50_words; duplicate_body_clean,
408,https://www.bbc.com/somali/articles/cd1mxrvxpggo,BBC Somali,Kulanka Ankara iyo wixii ay ka wada hadleen Shariif iyo Farmaajo,101,duplicate_body_clean,"Labo madaxweyne oo hore, Maxamed Cabdulaahi Farmaajo iyo Shariif Sheekh Axmed ayaa toddobaadkan ku kulmay magaalada Ankara ee xarunta u ah dowladda Turkey.\nWarar kala duwan ay..."
415,https://www.bbc.com/somali/articles/c4nqkpy9jpzo,BBC Somali,Kulanka Ankara iyo wixii ay ka wada hadleen Shariif iyo Farmaajo,101,duplicate_body_clean,"Labo madaxweyne oo hore, Maxamed Cabdulaahi Farmaajo iyo Shariif Sheekh Axmed ayaa toddobaadkan ku kulmay magaalada Ankara ee xarunta u ah dowladda Turkey.\nWarar kala duwan ay..."
421,https://www.bbc.com/somali/articles/ckvz4ejwelzo,BBC Somali,Daawo: Maxay Puntland iyo Dawladda Federaalka marwalba isku qabtaan?,0,empty_body_after_cleaning; body_under_50_words; duplicate_body_clean,
904,https://goobjoog.com/2026/03/24/maxkamadda-ciidanka-soomaaliya-xukun-ku-riday-ninkii-basaasay-feeryahanad-ramla-cali-iyo-kooxdiisa-2/,Goobjoog,Maxkamadda ciidanka Soomaaliya Xukun ku riday Ninkii Basaasay Feeryahanad Ramla Cali iyo Kooxdiisa,386,duplicate_body_clean,"Cabdisalaam Maxamed Xasan, Cabdullahi Cusmaan Maxamed, Cabdinaasir Cumar Axmed Subeyr, Faadumo Cismaan Subeyr Cali iyo Leylo Seef Faarax Axmed, Ayaa lagu Eedeeyay in ay siyaabo..."
1128,https://goobjoog.com/2025/10/25/ma-muqdisho-ayaa-keyn-noqatay-mise-geelii-ayaa-dab-joog-noqday/,Goobjoog,Ma Muqdisho ayaa keyn noqatay mise geelii ayaa dab joog noqday ?,299,duplicate_body_clean,"Waddooyinka caasimadda Soomaaliya ee Muqdisho, oo markii hore la ciir ciirayay lo'da, dameeraha iyo ariga buux dhaafiyay, ayaa haatan u muuqda kuwo isu beddelay goobo lagu dhaq..."
1164,https://goobjoog.com/2025/12/23/degdeg/,Goobjoog,Degdeg,30,body_under_50_words; headline_under_3_words,"Turkiga oo xaqiijiyay in dhinteen dhammaan dadkii saarnaa diyaaraddii la socday taliyaha ciidamada Liibiya General Mohamed Ali Ahmed, waxayna tirada dadkaasi ka koobnayd 5 qof ..."
1166,https://goobjoog.com/2025/11/24/geelii-dab-joogta-kamid-noqday-iyo-waddooyinka-muqdisho-oo-noqday-sida-keymo-geela-lagu-dhaqdo/,Goobjoog,geelii dab-joogta kamid noqday iyo waddooyinka Muqdisho oo noqday sida keymo geela lagu dhaqdo.,299,duplicate_body_clean,"Waddooyinka caasimadda Soomaaliya ee Muqdisho, oo markii hore la ciir ciirayay lo'da, dameeraha iyo ariga buux dhaafiyay, ayaa haatan u muuqda kuwo isu beddelay goobo lagu dhaq..."
1188,https://goobjoog.com/2026/03/24/maxkamadda-ciidanka-soomaaliya-xukun-ku-riday-ninkii-basaasay-feeryahanad-ramla-cali-iyo-kooxdiisa/,Goobjoog,Maxkamadda ciidanka Soomaaliya Xukun ku riday Ninkii Basaasay Feeryahanad Ramla Cali iyo Kooxdiisa,386,duplicate_body_clean,"Cabdisalaam Maxamed Xasan, Cabdullahi Cusmaan Maxamed, Cabdinaasir Cumar Axmed Subeyr, Faadumo Cismaan Subeyr Cali iyo Leylo Seef Faarax Axmed, Ayaa lagu Eedeeyay in ay siyaabo..."


## 14. Create the model-ready text columns

For the new project feature, the output contains both category and headline.

In [21]:
df["input_text"] = "analyze somali news article: " + df["body_clean"]
df["target_text"] = "category: siyaasad | headline: " + df["headline_clean"]

display(df[["input_text", "target_text"]].head(3))

,input_text,target_text
0,"analyze somali news article: Dawladdu waxay ka fikiraysaa soo saarista sharci lagaga saarayo Andrew Mountbatten-Windsor kaa oo ku aadan safka dhaxalka boqortooyada, sida ayBBC ...",category: siyaasad | headline: Dowladda Ingiriiska oo ka fikiraysa in Andrew laga saaro nidaamka boqortooyada
1,"analyze somali news article: Sanadihii u dambeeyay, jawiga gobolka Bariga dhexe waxa ahaa mid muujinayey khilaaf iyo kala duwanaasho mowqifyada iyo siyaasadaha Boqortooyada Suc...",category: siyaasad | headline: Khilaafka qarsoon ee Imaaraadka iyo Sucuudiga: ma noqon doonaa dhammaadka saaxiibtinnimadooda?
2,"analyze somali news article: Gen Muhoozi Kainerugaba oo ah nin si kulul wax ugu qora barta Twitter-ka (X) isla markaana madax u ah milatariga Uganda, ayaa dadka dhaliila waxay ...",category: siyaasad | headline: Wiilka Museveni oo ku hanjabay inuu dhufaani doono hoggaamiyaha mucaaradka


## 15. Create cleaned datasets

This notebook creates three useful outputs:

1. `siyaasad_cleaned_for_review.xlsx` - full cleaned file including review flags.
2. `siyaasad_model_ready.csv` - model-ready rows only.
3. `siyaasad_dropped_or_review_rows.xlsx` - rows excluded from model-ready file for manual review.

Default model-ready rule:

- remove empty body/headline
- remove exact duplicate `body_clean`
- remove articles under 50 words


In [22]:
model_ready = df.copy()

valid_mask = (
    model_ready["body_clean"].str.strip().ne("") &
    model_ready["headline_clean"].str.strip().ne("") &
    (model_ready["body_word_count_clean"] >= 50)
)

model_ready = model_ready[valid_mask].copy()
model_ready = model_ready.drop_duplicates(subset=["body_clean"], keep="first").copy()
model_ready = model_ready.drop_duplicates(subset=["url"], keep="first").copy()

dropped_or_review = df[~df.index.isin(model_ready.index)].copy()

final_cols = [
    "url",
    "headline_clean",
    "body_clean",
    "category",
    "source",
    "input_text",
    "target_text",
    "body_word_count_raw",
    "body_word_count_clean",
    "headline_word_count_clean",
    "needs_review",
    "review_reason",
]

review_cols = [
    "url",
    "headline",
    "headline_clean",
    "body",
    "body_clean",
    "category",
    "source",
    "input_text",
    "target_text",
    "body_word_count_raw",
    "body_word_count_clean",
    "headline_word_count_clean",
    "needs_review",
    "review_reason",
]

df[review_cols].to_excel(OUTPUT_REVIEW_FILE, index=False)
model_ready[final_cols].to_csv(OUTPUT_FINAL_CSV, index=False, encoding="utf-8-sig")
model_ready[final_cols].to_excel(OUTPUT_FINAL_XLSX, index=False)
dropped_or_review[review_cols].to_excel(OUTPUT_DROPPED_FILE, index=False)

print("Full cleaned review file:", OUTPUT_REVIEW_FILE)
print("Model-ready CSV:", OUTPUT_FINAL_CSV)
print("Model-ready Excel:", OUTPUT_FINAL_XLSX)
print("Dropped/review rows:", OUTPUT_DROPPED_FILE)
print("\nRaw rows:", len(df))
print("Model-ready rows:", len(model_ready))
print("Dropped or review rows:", len(dropped_or_review))
print("\nModel-ready source counts:")
print(model_ready["source"].value_counts())

Full cleaned review file: siyaasad_cleaned_for_review.xlsx
Model-ready CSV: siyaasad_model_ready.csv
Model-ready Excel: siyaasad_model_ready.xlsx
Dropped/review rows: siyaasad_dropped_or_review_rows.xlsx

Raw rows: 1692
Model-ready rows: 1685
Dropped or review rows: 7

Model-ready source counts:
source
Goobjoog      1247
BBC Somali     438
Name: count, dtype: int64
